In [1]:

#version 3 of test_single_trial_RAM_DISK.py with updated VOLPY from old single trial .py file
#version 4 after new work on correlation maps and volpy filtering
#version 5 to refine new corrleation map and implement, also to add peak width/height filtering and cell grid allignments that will be used in .py file
# TO RUN: conda activate caiman
# # python C:\Users\ICNLab\caiman_data\test_single_trial_RAM_DISK_3.py C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm

#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1.tsm'
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV3_T3new\FOV3_T3.tsm'
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'




froot = r'C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B'
import os
import re

# parser = argparse.ArgumentParser()
# parser.add_argument("froot", help="Path to the input movie file")
# args = parser.parse_args()

# froot = args.froot
#find all folders in r'C:\caiman_data\testdata\testdata\NF107.6B' and make list of those folder paths     

folder_paths = []
base_path = froot
for root, dirs, files in os.walk(base_path):
    for dir_name in dirs:
        folder_paths.append(os.path.join(root, dir_name))

# regex for folders like FOV1_T1, FOV12_T3, etc.
pattern = re.compile(r"^FOV\d+_T\d+$")

matching_folders = []
print("Searching for matching folders...")

for folder in folder_paths:
    for item in os.listdir(folder):
        item_path = os.path.join(folder, item)
        if os.path.isdir(item_path) and pattern.match(item):
            matching_folders.append(item_path)

print(matching_folders)



Searching for matching folders...
['C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T1', 'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T2', 'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250519\\FOV1_T1']


In [3]:


from collections import defaultdict
import os
import re

# group folders by FOV number
fov_groups = defaultdict(list)

for path in matching_folders:
    folder_name = os.path.basename(path)
    match = re.match(r"^FOV(\d+)_T(\d+)$", folder_name)
    if match:
        fov_number = match.group(1)  # e.g. "1" from FOV1_T2
        fov_groups[fov_number].append(path)

# # optional: sort each FOV group by T number
# for fov in fov_groups:
#     fov_groups[fov].sort(
#         key=lambda p: int(re.search(r"_T(\d+)$", os.path.basename(p)).group(1))
#     )

# sort each FOV group by date and T number
for fov in fov_groups:
    fov_groups[fov].sort(
        key=lambda p: (
            int(os.path.basename(os.path.dirname(p))),  # date: 20250505
            int(re.search(r"_T(\d+)$", os.path.basename(p)).group(1))  # trial number
        )
    )

print("Grouped folders by FOV:")
print(fov_groups)

for fov, paths in sorted(fov_groups.items(), key=lambda x: int(x[0])):
    print(f"Analyzing FOV{fov} with {len(paths)} sessions")
    previous_folder_path = ""
    #analyzeFOV(paths, previous_folder_path)



Grouped folders by FOV:
defaultdict(<class 'list'>, {'1': ['C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T1', 'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T2', 'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250519\\FOV1_T1']})
Analyzing FOV1 with 3 sessions


In [3]:
#def analyzeFOV(folder_paths, previous_folder_path):
print("Importing packages and Initializing...")
from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from pathlib import Path
from PIL import Image

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd
import mmap



from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc


logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)




##BEGIN MAIN ANALYSIS LOOP
for folder_path in paths:
    print("Analyzing folder:", folder_path)



Importing packages and Initializing...
Analyzing folder: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1
Analyzing folder: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250519\FOV1_T1
Analyzing folder: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T2


In [4]:
folder_path = paths[1]  # Example folder path
print("Analyzing folder:", folder_path)
previous_folder_path = paths[0] 
print("Previous folder:", previous_folder_path)

Analyzing folder: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250519\FOV1_T1
Previous folder: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1


In [5]:
# find the .tsm file in the folder
tsm_files = [f for f in os.listdir(folder_path) if f.endswith('.tsm')]
if not tsm_files:
    print(f"No .tsm file found in {folder_path}, skipping.")
    #continue
#continue if more than one .tsm file found
if len(tsm_files) > 1:
    print(f"Multiple .tsm files found in {folder_path}, skipping.")
    #continue
fname = os.path.join(folder_path, tsm_files[0])
print("Processing file:", fname)



Processing file: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250519\FOV1_T1\FOV1_T1.tsm


In [6]:


##
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'
fr = 640
print(fname, fr)

import matplotlib
print(matplotlib.get_backend())



C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250519\FOV1_T1\FOV1_T1.tsm 640
qtagg


In [7]:

##
# Cleanup R:/ drive
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")



Cleaning up R:/ drive...
Cleared all files from R:/


In [8]:

##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

##
print("Loading data...")
m_orig = cm.load(fname)
#ds_ratio = 0.2

##
c, dview, n_processes = cm.cluster.setup_cluster(
            backend='local', n_processes=None, single_thread=False)

##
print("Motion correction...")
mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True)
#about 2.3 minutes for 12800 frames (2m 13-21 s)
print("Done.")



Loading data...
Motion correction...
Saving mmap to:  R:/FOV1_T1_rig__d1_512_d2_512_d3_1_order_F_frames_19200.mmap
Done.


In [9]:

##
print("Loading corrected movie...")
m_rig = cm.load(mc.mmap_file) # 11s
ds_ratio = 0.2
print("Done.")


Loading corrected movie...


100%|██████████| 1/1 [00:12<00:00, 12.22s/it]


Done.


In [10]:
del m_orig
gc.collect()

9

In [11]:
#CONVERT ORDER FROM C TO F VIA DATA STREAMING

from pathlib import Path
import re

name = Path(mc.mmap_file[0]).name

Y = int(re.search(r'_d1_(\d+)', name).group(1))
X = int(re.search(r'_d2_(\d+)', name).group(1))
T = int(re.search(r'_frames_(\d+)', name).group(1))

shape = (T, Y, X)


src = np.memmap(
    mc.mmap_file[0],
    dtype='float32',
    mode='r+',
    shape=(T, Y, X),
    order='C'   # matches physical layout
)

print("Saving stabilized movie to RAM-disk...")
# Path to RAM disk memmap
p = Path(fname)
ram_path = Path(r'R:/') / f"{p.stem}_rig__d1_{m_rig.shape[1]}_d2_{m_rig.shape[2]}_d3_1_order_C_frames_{m_rig.shape[0]}.mmap"
ram_path = str(ram_path).replace("/", "\\") 

dst = np.memmap(
    ram_path,
    dtype='float32',
    mode='w+',
    shape=(T, Y, X),
    order='F'   # true pixel-wise contiguous time
)

chunk = 16  # tune this

for t0 in range(0, T, chunk):
    t1 = min(t0 + chunk, T)

    block = src[t0:t1]      # small buffer
    dst[t0:t1] = block     # repacked to F order

    # optional but recommended on RAM disk
    #src[t0:t1] = 0.0       # free backing pages
    
dst.flush()
del src, dst



Saving stabilized movie to RAM-disk...


In [18]:

##

print("Computing mean and correlation images...")
img = np.mean(m_rig, axis=0)
img = (img-np.mean(img))/np.std(img)


import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from tqdm import tqdm

# ===============================
# 1. Parameters
# ===============================
SHAPE = (12800, 512, 512)
BANDPASS = (5, 300) # Hz #70,300


# ===============================
# 2. Load memory-mapped video
# ===============================

video = np.memmap(
    mc.mmap_file[0],
    dtype=np.float32,
    mode="r",
    shape=SHAPE,
    order="C"
).swapaxes(1, 2)

T, H, W = video.shape
print(f"Loaded video: {video.shape}")

# ===============================
# 3. High-pass filter (bandpass-compatible API)
# ===============================
def bandpass_filter(data, fs, low, high=None, order=3):
    """
    High-pass filter using the 'low' cutoff.
    The 'high' argument is accepted for API compatibility but ignored.
    """
    nyq = 0.5 * fs
    b, a = butter(order, low / nyq, btype="high")
    return filtfilt(b, a, data, axis=0)

# ===============================
# Parameters
# ===============================
TILE_SIZE = 4
H, W = 512, 512
FRAME_RATE = fr
# (low, high), high ignored
DISPLAY_CLIP = 99

# ===============================
# Coherence metric
# ===============================
def coherence_metric(tile_filt):
    """
    tile_filt: shape (T, Npix)
    Returns mean pixel-to-tile correlation.
    """
    # Tile reference (subthreshold signals sum coherently)
    ref = tile_filt.mean(axis=1)


    ref -= ref.mean()
    ref_std = ref.std() + 1e-9

    # Normalize reference
    ref /= ref_std

    # Normalize pixels
    pix = tile_filt - tile_filt.mean(axis=0)
    pix /= (pix.std(axis=0) + 1e-9)

    # Correlation with reference
    corr = np.mean(ref[:, None] * pix, axis=0)

    # Use mean absolute correlation as coherence
    return np.mean(np.abs(corr))


# ===============================
# Output tile map
# ===============================
n_tiles_y = H // TILE_SIZE
n_tiles_x = W // TILE_SIZE

tile_coherence_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# ===============================
# Main loop
# ===============================
with tqdm(total=n_tiles_y * n_tiles_x, desc="Computing coherence") as pbar:
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):

            y0 = ty * TILE_SIZE
            y1 = y0 + TILE_SIZE
            x0 = tx * TILE_SIZE
            x1 = x0 + TILE_SIZE

            # Extract tile: (T, 16, 16)
            tile = video[:, y0:y1, x0:x1]
            tile = tile.reshape(T, -1)

            # High-pass filter all pixels independently
            tile_filt = bandpass_filter(
                tile, FRAME_RATE, *BANDPASS
            )

            # Compute coherence
            tile_coherence_map[ty, tx] = coherence_metric(tile_filt)

            pbar.update(1)

# ===============================
# Expand to image resolution
# ===============================
coherence_image = np.repeat(
    np.repeat(tile_coherence_map, TILE_SIZE, axis=0),
    TILE_SIZE, axis=1
)

# ===============================
# Visualization
# ===============================
vmax = np.percentile(coherence_image, DISPLAY_CLIP)

plt.figure(figsize=(6, 6))
plt.imshow(coherence_image, cmap="viridis", vmin=0, vmax=vmax)
plt.title("Grid-based subthreshold coherence ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
plt.colorbar(label="Mean |pixel–tile correlation|")
plt.axis("off")
plt.tight_layout()
plt.show()
plt.close()

img_corr = coherence_image
summary_images = np.stack([img, img, img_corr], axis=0).astype(np.float32)
cm.movie(summary_images).save(fname[:-5]+'_summary_images.tif')

plt.imshow(summary_images[0], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_mean.tif', format='tif', bbox_inches='tight', pad_inches=0)


plt.imshow(summary_images[2], cmap='gray')
plt.axis('off')
plt.savefig(fname[:-4]+'_corr.tif', format='tif', bbox_inches='tight', pad_inches=0)
img = summary_images.transpose([1, 2, 0])


print(fname[:-4]+'_corr.tif')
height, width = img.shape[:2]
print(img.shape)

# --------------------------------------------------------------
# Extract channels like MATLAB
# --------------------------------------------------------------
R = img[:, :, 0]
B = img[:, :, 2]

# --------------------------------------------------------------
# MATLAB-style normalization (mat2gray + uint8)
# --------------------------------------------------------------
def normalize_like_matlab(x):
    x = x.astype(np.float64)
    mn = x.min()
    mx = x.max()
    x = (x - mn) / (mx - mn + 1e-12)

    # MATLAB uint8 applies rounding, not floor
    x = np.round(255 * x).astype(np.uint8)
    return x

R_norm = normalize_like_matlab(R)
B_norm = normalize_like_matlab(B)

# --------------------------------------------------------------
# Build MATLAB-equivalent RGB (R,R,B)
# --------------------------------------------------------------
rgb = np.stack([R_norm, R_norm, B_norm], axis=2).astype(np.uint8)

# --------------------------------------------------------------
# Save as PNG (MATLAB-compatible pixel data)
# --------------------------------------------------------------
outname = fname[:-4] + "_py.png"
Image.fromarray(rgb).save(outname)

print("Saved:", outname)
img = rgb.copy()



##
print("Running Mask R-CNN inference...")
weights_path="C:/Users/ICNLab/caiman_data/testdata/testdata/mask_rcnn_neuron_0012.h5"
#download_model('mask_rcnn')
#ROIs, r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
r = utils.mrcnn_inference(img, size_range=[0, 40], weights_path=weights_path, display_result=True)
ROIs = r['masks'].transpose([2, 0, 1])
Coords = r['rois']
cm.movie(ROIs).save(fname[:-4]+'newmrcnn_ROIs.hdf5')

fig, axs = plt.subplots(1, 2)
axs[0].imshow(summary_images[1])
axs[1].imshow(ROIs.sum(0))
axs[0].set_title('mean image')
axs[1].set_title('masks')
plt.savefig(fname[:-6] + 'newmrcnn_ROIs.png', format='png', bbox_inches='tight', pad_inches=0)

print("Segmentation Completed.")

cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False, maxtasksperchild=1)




Computing mean and correlation images...
Loaded video: (12800, 512, 512)


Computing coherence: 100%|██████████| 16384/16384 [01:59<00:00, 137.49it/s]


C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250519\FOV1_T1\FOV1_T1_corr.tif
(512, 512, 3)
Saved: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250519\FOV1_T1\FOV1_T1_py.png
Running Mask R-CNN inference...

Configurations:
BACKBONE                       resnet50
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        200
DETECTION_MIN_CONFIDENCE       0
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  512
IMAGE_META_SIZE                14
IMAGE_MIN_DIM                  512
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              crop
IMAGE_SHAPE                    [512 512   3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE       

     1227653 [deprecation.py:            new_func():554][18172] From c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\tensorflow\python\util\deprecation.py:629: calling map_fn_v2 (from tensorflow.python.ops.map_fn) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Use fn_output_signature instead


Processing 1 images
image                    shape: (512, 512, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min:  -91.11000  max:  168.24000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  512.00000  int32
anchors                  shape: (1, 65472, 4)         min:   -0.04428  max:    1.01297  float32


c:\Users\ICNLab\anaconda3\envs\caiman\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


MADE FIGURE
Segmentation Completed.


In [19]:

##
ROIs = ROIs                                   # region of interests
index = list(range(len(ROIs)))                # index of neurons
weights = None                                # if None, use ROIs for initialization; to reuse weights check reuse weights block

template_size = 0.008                         # half size of the window length for spike templates, default is 20 ms
context_size = 35                             # number of pixels surrounding the ROI to censor from the background PCA
visualize_ROI = False                         # whether to visualize the region of interest inside the context region
hp_freq_pb = 1 / 3                            # parameter for high-pass filter to remove photobleaching
clip = 100                                    # maximum number of spikes to form spike template
threshold_method = 'simple'                   # adaptive_threshold or simple
min_spikes= 10                                # minimal spikes to be found
pnorm = 0.5                                   # a variable deciding the amount of spikes chosen for adaptive threshold method
threshold = 5                                 # threshold for finding spikes only used in simple threshold method, Increase the threshold to find less spikes
do_plot = False                               # plot detail of spikes, template for the last iteration
ridge_bg= 0.05                                # ridge regression regularizer strength for background removement, larger value specifies stronger regularization
sub_freq = 20                                 # frequency for subthreshold extraction
weight_update = 'ridge'                       # ridge or NMF for weight update
n_iter = 2                                    # number of iterations alternating between estimating spike times and spatial filters
censor_size = 5                               # size of the censoring region around the ROI

opts_dict={'fnames': ram_path,   #'fnames': fname_new,
        'ROIs': ROIs,
        'index': index,
        'weights': weights,
        'template_size': template_size,
        'context_size': context_size,
        'visualize_ROI': visualize_ROI,
        'hp_freq_pb': hp_freq_pb,
        'clip': clip,
        'threshold_method': threshold_method,
        'min_spikes':min_spikes,
        'pnorm': pnorm,
        'threshold': threshold,
        'do_plot':do_plot,
        'ridge_bg':ridge_bg,
        'sub_freq': sub_freq,
        'weight_update': weight_update,
        'n_iter': n_iter,
        'censor_size': censor_size}

opts.change_params(params_dict=opts_dict)

vpy = VOLPY(n_processes=n_processes, dview=dview, params=opts)

print("Running VOLPY fit...")
vpy.fit(n_processes=n_processes, dview=dview)
#takes a while to run
print("Done.")


Running VOLPY fit...
Starting VOLPY spike detection...
Done.


In [52]:

# Visualize spatial footprints and traces
# print(np.where(vpy.estimates['locality'])[0])    # neurons that pass locality test
# idx = np.where(vpy.estimates['locality'] > 0)[0]
# utils.view_components(vpy.estimates, img_corr, idx)

##

# Reconstructed movie
# flip_signal = True    
# mv_all = utils.reconstructed_movie(vpy.estimates.copy(), fnames=mc.mmap_file,
#                                         idx=idx, scope=(0,1000), flip_signal=flip_signal)
#mv_all.play(fr=40, magnification=3)

##

# #save ROIs as npy array
# np.save(fname[:-4]+'newmrcnn_ROIs.npy', ROIs)
# print("Saved ROIs as npy array:", fname[:-4]+'newmrcnn_ROIs.npy')

###NEW SECTION FOR ROI COORDINATE EXTRACTION
cell_centers = [((y1 + y2) // 2, (x1 + x2) // 2) for (y1, x1, y2, x2) in Coords]
cell_centers = np.array(cell_centers)
print("Cell centers:", cell_centers)    
#display the cell centers on the image
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img, cmap='gray') # Display the image
ax.scatter(cell_centers[:, 1], cell_centers[:, 0], color='red') # Display the cell centers
ax.set_title('Cell centers')    # Set the title of the plot
plt.savefig(fname[:-4] + '_cell_centers.png', format='png', bbox_inches='tight', pad_inches=0)
#plt.close() # Save the figure and close the plot     

# Save to a file
save_path = fname[:-4] + '_cell_centers.npy'
np.save(save_path, cell_centers)

print(f"Cell centers saved to {save_path}")
print(len(cell_centers))



Cell centers: [[392 483]
 [ 10 351]
 [311 250]
 [208 181]
 [408  76]
 [ 61  36]
 [101 270]
 [201 105]
 [250 226]
 [217 432]
 [391 311]
 [467 291]
 [286  51]
 [ 44 366]
 [106 131]
 [ 14 415]
 [121 473]
 [ 42 395]
 [447 397]
 [114 306]
 [279 366]
 [103 331]
 [194 378]
 [201  80]
 [218 122]
 [ 22 478]
 [211  47]
 [426 479]
 [339  30]
 [382 197]
 [161 173]
 [345  55]
 [ 92 412]
 [ 34 250]
 [187 120]
 [ 89 464]
 [180 496]
 [328 404]
 [ 49  95]
 [167  91]
 [228  92]
 [123 455]
 [126 324]
 [ 40 293]
 [166 178]
 [282 240]
 [365 346]
 [126  80]
 [478  79]
 [ 87 174]
 [319 291]
 [267 249]
 [216 296]
 [264 154]
 [106 368]
 [ 95 141]
 [ 78 202]
 [372 334]
 [ 45 170]
 [ 89 436]
 [ 79 151]
 [117 364]
 [ 60 221]
 [107 454]
 [112 236]
 [266 205]
 [ 54 481]
 [426  45]
 [166  76]
 [118 349]
 [ 25 255]
 [290 252]
 [ 97 305]
 [ 95 114]
 [ 83 256]
 [ 66 163]
 [304 149]
 [122  48]
 [369 429]
 [410 366]
 [ 22 216]
 [ 97 447]
 [116 212]
 [ 83 494]
 [113 509]
 [201  16]
 [ 84 233]
 [219 316]
 [316 401]
 [398 3

In [53]:
print(previous_folder_path)

vpy.estimates['ROIs'] = ROIs
vpy.estimates['Coords'] = Coords
vpy.estimates['cell_centers'] = cell_centers
vpy.estimates['Cell_IDs'] = np.zeros(len(ROIs))



C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1


In [54]:

##
vpynew = vpy.estimates
vpynew['spikes'] = np.array(vpynew['spikes'], dtype=object)

num_frames = np.max(vpynew['dFF'].shape)
dur = num_frames/640
vpynew['snr_over_3'] = []

vpynew['raster'] = np.zeros_like(vpynew['dFF'])
vpynew['firing_rate'] = np.zeros_like(vpynew['dFF'])
vpynew['unique_trace'] = []
vpynew['cell_idxs'] = []

for i in range(vpynew['dFF'].shape[0]-1):
    vpynew['raster'][i, vpynew['spikes'][i]] = 1
    vpynew['firing_rate'][i] = savgol_filter(np.convolve(vpynew['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

for i in range(len(vpynew['Cell_IDs'])):
    vpynew['snr_over_3'].append(vpynew['snr'][i] > 3.0)

print("Number of neurons with SNR > 3:", np.sum(vpynew['snr_over_3']))

print(vpynew['Cell_IDs'])

if np.sum(vpynew['snr_over_3']) > 0:
    to_remove = set()
    dFF = np.array(vpynew['dFF']).astype(float)
    R = np.corrcoef(dFF)
    idx0, idx1 = np.where(np.triu(R, 1) > 0.9)
    max_vals = np.max(dFF, axis=1)
    smaller = np.where(max_vals[idx0] < max_vals[idx1], idx0, idx1)
    to_remove.update(smaller.tolist())
    vpynew['unique_trace'] = [True if x not in to_remove else False for x in range(len(vpynew['Cell_IDs']))]

    # dFF = np.array(vpynew['dFF']).astype(float)
    # R = np.corrcoef(dFF)
    # r = np.array(np.where(np.triu(R,1)>0.7))
    # for i in range(0,r.shape[1]):
    #     if np.max(dFF[r[0][i]]) < np.max(dFF[r[1][i]]):
    #         r[1][i] = r[0][i]

    # vpynew['cellID'] = [x for x in vpynew['cellID'] if x not in r[1]]
print(vpynew['unique_trace'])
print("There are", np.sum(vpynew['unique_trace']), "unique traces after correlation filtering.")
print("And there were ", len(to_remove), "traces removed due to high correlation.")
vpynew['cell_idxs'] = []
for cell in range(len(vpynew['Cell_IDs'])):
    if vpynew['snr_over_3'][cell] and vpynew['unique_trace'][cell]:
        vpynew['cell_idxs'].append(cell)

print("Final number of cells after SNR and correlation filtering:", len(vpynew['cell_idxs']))
print(vpynew['cell_idxs'])
print(len(vpynew['cell_idxs']))

Number of neurons with SNR > 3: 11
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0.]
[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
There are 98 unique traces after correlation filtering.
And there were  0 tr

In [55]:
cell_centers_all = cell_centers
#cell_centers = cell_centers[vpynew['cell_idxs']]

vpy.estimates['cell_centers'] = cell_centers_all
vpy.estimates['cell_idxs'] = vpynew['cell_idxs']

#If first file in FOV, assign Cell_IDs as 1,2,3,... to each cell in cell_idxs
if previous_folder_path == "":
    for i in len(vpynew['cell_idxs']):
        vpy.estimates['Cell_IDs'][vpynew['cell_idxs'][i]] = i+1
    print(f"First session for this FOV. Assigned Cell_IDs from 1 to {len(vpynew['cell_idxs'])}.")

In [56]:
os.path.join(previous_folder_path, previous_folder_path.split('\\')[-1] + 'volpy.npy')

'C:\\Users\\ICNLab\\caiman_data\\testdata\\testdata\\NF107.6B\\20250505\\FOV1_T1\\FOV1_T1volpy.npy'

In [95]:
plt.close('all')

In [ ]:
#RANSAC ALLIGNMENT WITH ADDED ROTATION

from scipy.spatial import cKDTree
import numpy as np
import random
from scipy.ndimage import affine_transform

# Load old cell centers if not first file
if previous_folder_path != "":
    old_save_name = os.path.join(
        previous_folder_path,
        previous_folder_path.split('\\')[-1] + 'volpy.npy'
    )

    if os.path.exists(old_save_name):
        old_estimates = np.load(old_save_name, allow_pickle=True).item()
        old_cell_centers = old_estimates['cell_centers']
        #old_cell_idxs = old_estimates['cell_idxs']
        #old_cell_centers = old_cell_centers[old_cell_idxs] for after testing allginment working
        old_cell_IDs = old_estimates['Cell_IDs']
        print(f"Loaded old cell centers from {old_save_name}")
    else:
        print(f"No previous VOLPY estimates found at {old_save_name}")
        old_cell_centers = None


# ---------------------------------------------------------
# RANSAC TRANSLATION ESTIMATION
# ---------------------------------------------------------

def rotation_matrix(angle):
    """Returns a 2D rotation matrix for a given angle in radians."""
    return np.array([[np.cos(angle), -np.sin(angle)], 
                     [np.sin(angle), np.cos(angle)]])



def ransac_alignment(A, B, n_iter=2000, inlier_thresh=15):
    """
    Estimate translation t and rotation matrix R such that A + R ≈ B
    A: new cell centers (N,2)
    B: old cell centers (M,2)
    """
    tree = cKDTree(B)
    best_t = np.zeros(2)
    best_R = np.eye(2)
    best_inliers = []

    for _ in range(n_iter):
        # Sample two random points from A and B
        a = A[random.randrange(len(A))]
        b = B[random.randrange(len(B))]

        # Compute translation vector
        t = b - a
        
        # Compute rotation angle based on the vector difference
        angle = np.arctan2(b[1] - a[1], b[0] - a[0])
        R = rotation_matrix(angle)
        
        # Apply translation and rotation to A
        A_transformed = np.dot(A, R.T) + t

        # Query the nearest neighbors from B
        dists, idx = tree.query(A_transformed)

        # Identify inliers (points with distance below threshold)
        inliers = np.where(dists < inlier_thresh)[0]

        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_t = t
            best_R = R

    # Refine translation and rotation using inliers
    if len(best_inliers) > 0:
        dists, idx = tree.query(np.dot(A[best_inliers], best_R.T) + best_t)
        refined_t = np.mean(B[idx] - A[best_inliers], axis=0)
        return refined_t, best_R, best_inliers

    return best_t, best_R, np.array([])

import numpy as np
import random
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

def select_points_within_radius(A, radius=50, num_points=5):
    """
    Select a set of points from A that are within the given radius of each other.
    A: new cell centers (N, 2)
    radius: maximum distance between points
    num_points: number of points to select
    """
    selected_points = []
    while len(selected_points) < num_points:
        # Randomly choose a point
        idx = random.randrange(len(A))
        point = A[idx]

        # Find other points within the specified radius
        distances = np.linalg.norm(A - point, axis=1)
        close_points_idx = np.where(distances <= radius)[0]

        # If there are enough points within the radius, select them
        if len(close_points_idx) >= num_points:
            selected_points = A[close_points_idx[:num_points]]
            
    return selected_points

def compute_relative_positions(points):
    """
    Compute the pairwise relative positions (distances and angles) between a set of points.
    points: (num_points, 2) - array of selected points
    Returns the pairwise distances and angles
    """
    num_points = len(points)
    distances = np.zeros((num_points, num_points))
    angles = np.zeros((num_points, num_points))

    for i in range(num_points):
        for j in range(i+1, num_points):
            # Compute the relative distance and angle between point i and point j
            delta = points[j] - points[i]
            distances[i, j] = np.linalg.norm(delta)
            distances[j, i] = distances[i, j]

            # Compute the angle
            angles[i, j] = np.arctan2(delta[1], delta[0])
            angles[j, i] = angles[i, j]
    
    return distances, angles

def ransac_alignment_using_relative_positions(A, B, num_points=5, radius=50, n_iter=10000, inlier_thresh=15):
    """
    Estimate translation and rotation using the relative positions of selected points.
    A: new cell centers (N, 2)
    B: old cell centers (M, 2)
    """
    best_t = np.zeros(2)
    best_R = np.eye(2)
    best_inliers = []

    for _ in range(n_iter):
        # Step 1: Select a random set of points within the specified radius
        selected_A = select_points_within_radius(A, radius, num_points)
        selected_B = select_points_within_radius(B, radius, num_points)

        # Step 2: Compute the relative positions (distances and angles)
        distances_A, angles_A = compute_relative_positions(selected_A)
        distances_B, angles_B = compute_relative_positions(selected_B)

        # Step 3: Compute the transformation (translation and rotation)
        # Compute translation by aligning centroids
        centroid_A = np.mean(selected_A, axis=0)
        centroid_B = np.mean(selected_B, axis=0)
        t = centroid_B - centroid_A

        # Compute rotation by minimizing the difference in angles
        angle_diff = np.mean(angles_B - angles_A)
        R = rotation_matrix(angle_diff)

        # Apply rotation and translation to the new points
        A_transformed = np.dot(A, R.T) + t

        # Step 4: Query nearest neighbors in B and find inliers
        tree = cKDTree(B)
        dists, idx = tree.query(A_transformed)
        inliers = np.where(dists < inlier_thresh)[0]

        if len(inliers) > len(best_inliers):   #MAY WANT TO CHANGE TO MORE PRECISE METRIC
            best_inliers = inliers
            best_t = t
            best_R = R

    return best_t, best_R, best_inliers


# ---------------------------------------------------------
# APPLY RANSAC ALIGNMENT
# ---------------------------------------------------------

from scipy.ndimage import rotate

if old_cell_centers is not None:
    # Perform the RANSAC alignment based on relative positions
    t, R, inliers = ransac_alignment_using_relative_positions(cell_centers, old_cell_centers)

    # Apply the translation and rotation to the new cell centers
    X_new_aligned = np.dot(cell_centers, R.T) + t

    # Visualize the result
    fig, ax = plt.subplots(figsize=(6, 6))



    # Inverse rotation (required by affine_transform)
    A = R.T

    # Inverse translation
    offset = -A @ np.array(t)
    # Simple average of channels
    img_gray = img.mean(axis=2)

    img_transformed = affine_transform(
        img_gray,
        A,
        offset=offset,
        order=1,               # bilinear interpolation
        mode="constant",       # zero padding
        cval=0.0
    )

    ax.imshow(img_transformed, cmap='gray')
    ax.scatter(
        old_cell_centers[:, 1], old_cell_centers[:, 0],
        color='blue', s=10, label='Old'
    )
    ax.scatter(
        X_new_aligned[:, 1], X_new_aligned[:, 0],
        color='red', s=10, label='New aligned'
    )
    #plot inliers with pink color
    ax.scatter(
        X_new_aligned[inliers, 1], X_new_aligned[inliers, 0],
        color='pink', s=20, label='Inliers'
    )
    ax.set_title('RANSAC Alignment with Rotation Included')
    ax.legend()
    plt.show()



Loaded old cell centers from C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1volpy.npy


In [ ]:
#ID region of best fit and use as a center of rotation

def get_best_fit_coords(cell_centers, old_cell_centers):
    #get distances to nearest neighbors with cKDTree
    tree = cKDTree(old_cell_centers)
    dists, idx = tree.query(cell_centers)

    #take top 20 closest points
    if len(dists) >= 20:
        best_fit_indices = np.argsort(dists)[:20]
    elif len(dists) > 0 and len(dists) < 20:
        best_fit_indices = np.argsort(dists)
    coords_ofbestfit = cell_centers[best_fit_indices]
    #weighted average of best fit coords
    center_of_rotation = np.average(
        coords_ofbestfit,
        axis=0,
        weights=1/(dists[best_fit_indices]+1e-9)
    )
    return coords_ofbestfit, center_of_rotation

def rotation_fitter(A, B, center):
    """
    Estimate rotation matrix R that aligns A with B around a center point.
    A: New cell centers (N, 2)
    B: Old cell centers (M, 2)
    center: point around which to rotate
    """
    # Center the points around the rotation center
    A_centered = A - center
    B_centered = B - center
    
    # Compute the SVD of the cross-covariance matrix
    H = A_centered.T @ B_centered
    U, _, Vt = svd(H)
    
    # Compute the rotation matrix
    R = Vt.T @ U.T
    
    # Ensure a proper rotation (determinant should be 1, fix reflection if necessary)
    if np.linalg.det(R) < 0:
        Vt[1, :] *= -1
        R = Vt.T @ U.T
    
    return R

def rotate_points(points, angle, center):
    R = rotation_matrix(angle)
    return (points - center) @ R.T + center

def rotation_fitter_regular(A, B, center):
    """
    Explore rotation space in regular intervals to find best rotation aligning A with B around center.
    Then iterate to refine exploration of angles about the best previous angle.
    A: New cell centers (N, 2)
    B: Old cell centers (M, 2)
    center: point around which to rotate
    """

    best_angle = 0
    best_inliers = []

    # Coarse search
    #only explore rotation angles from -10 to 10 degrees (360) in 1 degree increments
    for angle_deg in range(-10, 11, 1):
        angle_rad = np.radians(angle_deg)
        A_rotated = rotate_points(A, angle_rad, center)

        # Find inliers based on nearest neighbors
        tree = cKDTree(B)
        dists, idx = tree.query(A_rotated)
        inliers = np.where(dists < 15)[0]

        if len(inliers) > len(best_inliers):  
            best_inliers = inliers
        #if np.median(dists[inliers]) < np.median(dists[best_inliers]) if len(best_inliers) > 0 else float('inf'):
            best_angle = angle_rad
            print("Best angle1:", best_angle)

    # Fine search around best angle
    for angle_deg in np.arange(np.degrees(best_angle)-1, np.degrees(best_angle)+1, 0.1):
        angle_rad = np.radians(angle_deg)
        A_rotated = rotate_points(A, angle_rad, center)

        # Find inliers based on nearest neighbors
        tree = cKDTree(B)
        dists, idx = tree.query(A_rotated)
        inliers = np.where(dists < 15)[0]

        if len(inliers) > len(best_inliers):  
            best_inliers = inliers
        #if np.median(dists[inliers]) < np.median(dists[best_inliers]) if len(best_inliers) > 0 else float('inf'):
            best_angle = angle_rad
            print("Best angle2:", best_angle)

    #one more finer fit
    for angle_deg in np.arange(np.degrees(best_angle)-0.1, np.degrees(best_angle)+0.1, 0.01):
        angle_rad = np.radians(angle_deg)
        A_rotated = rotate_points(A, angle_rad, center)

        # Find inliers based on nearest neighbors
        tree = cKDTree(B)
        dists, idx = tree.query(A_rotated)
        inliers = np.where(dists < 15)[0]

        #if np.median(dists[inliers]) < np.median(dists[best_inliers]) if len(best_inliers) > 0 else float('inf'):
        if len(inliers) > len(best_inliers):  
            best_inliers = inliers
        #if np.median(dists[inliers]) < np.median(dists[best_inliers]) if len(best_inliers) > 0 else float('inf'):
            best_angle = angle_rad
            print("Best angle3:", best_angle)


    #one more finer fit
    for angle_deg in np.arange(np.degrees(best_angle)-0.1, np.degrees(best_angle)+0.1, 0.01):
        angle_rad = np.radians(angle_deg)
        A_rotated = rotate_points(A, angle_rad, center)

        # Find inliers based on nearest neighbors
        tree = cKDTree(B)
        newdists, idx = tree.query(A_rotated)
        inliers = np.where(dists < 15)[0]

        if (np.mean(newdists[best_inliers]) < np.mean(dists[best_inliers]) if len(best_inliers) > 0 else True) and (len(inliers) >= len(best_inliers)):
            best_angle = angle_rad
            print("Best angle4:", best_angle)
            dists=newdists

    return best_angle, best_inliers


# --- Apply rotation fitting ---

if old_cell_centers is not None:
    coords_ofbestfit, center_of_rotation = get_best_fit_coords(
        X_new_aligned,
        old_cell_centers
    )

    best_angle, inliers = rotation_fitter_regular(
        X_new_aligned,
        old_cell_centers,
        center_of_rotation
    )

    # Apply rotation to all new cell centers
    X_new_rotated = rotate_points(
        X_new_aligned,
        best_angle,
        center_of_rotation
    )

    # Visualize the result
    fig, ax = plt.subplots(figsize=(6, 6))

    A2 = R.T @ R2.T    
    t2 = R2 @ np.array(t) - R2 @ np.array(center_of_rotation) + np.array(center_of_rotation)
    offset = -A2 @ t2

    img_transformed = affine_transform(
        img_gray,
        A2,
        offset=offset,
        order=1,               # bilinear interpolation
        mode="constant",       # zero padding
        cval=0.0
    )

    ax.imshow(img_transformed, cmap='gray')
    ax.scatter(
        old_cell_centers[:, 1], old_cell_centers[:, 0],
        color='blue', s=10, label='Old'
    )
    ax.scatter(
        X_new_rotated[:, 1], X_new_rotated[:, 0],
        color='red', s=10, label='New aligned'
    )
    #plot inliers with pink color
    ax.scatter(
        X_new_rotated[inliers, 1], X_new_rotated[inliers, 0],
        color='pink', s=20, label='Inliers'
    )
    ax.set_title('FINE ROTATION WITH NEW CENTER')
    ax.legend()
    plt.show()



Best angle1: -0.17453292519943295
Best angle1: -0.15707963267948966
Best angle1: -0.12217304763960307
Best angle1: -0.10471975511965978
Best angle1: -0.08726646259971647
Best angle1: -0.06981317007977318
Best angle1: -0.05235987755982989
Best angle1: -0.03490658503988659
Best angle1: -0.017453292519943295
Best angle1: 0.0
Best angle1: 0.017453292519943295
Best angle2: 0.024434609527920616


In [ ]:
#Simple translation grid search

def translation_fitter_regular(A, B):
    """
    Explore rotation space in regular intervals to find best rotation aligning A with B around center.
    Then iterate to refine exploration of angles about the best previous angle.
    A: New cell centers (N, 2)
    B: Old cell centers (M, 2)
    center: point around which to rotate
    """

    best_shift= 0
    best_inliers = []

    # Coarse search
    #only translations in 1 pixel increments

    tree = cKDTree(B)
    dists, idx = tree.query(A)
    alldists=[]
    for xshift in range(-15, 16, 1):
        for yshift in range(-15, 16, 1):
            n = len(A)
            shift_vector = np.tile([xshift, yshift], (n, 1))
            #apply shift
            A_withshift = A + shift_vector

            # Find inliers based on nearest neighbors
            tree = cKDTree(B)
            newdists, idx = tree.query(A_withshift)
            inliers = np.where(dists < 15)[0]
            alldists.append(np.mean(newdists))
            # if xshift==-5 & yshift==0:
            #     print("Mean dist at -5, 0", np.mean(dists))
            if len(inliers) > len(best_inliers):  
                best_inliers = inliers
            if (np.mean(newdists[best_inliers]) < np.mean(dists[best_inliers]) if len(best_inliers) > 0 else True):
                best_shift = shift_vector
                dists=newdists
                print("Mean dist", np.mean(dists))
                print("Best shfit1:", shift_vector[0])
    return best_shift, best_inliers, alldists

# --- Apply translation fitting ---

if old_cell_centers is not None:

    best_shift, inliers, alldists = translation_fitter_regular(
        X_new_rotated,
        old_cell_centers
    )
    # n = len(X_new_rotated)
    # best_shift = np.tile([-5, 0], (n, 1))


    alldists_array = np.array(alldists)

    xrange = np.arange(-15, 16)
    yrange = np.arange(-15, 16)
    alldists_2d = alldists_array.reshape(len(xrange), len(yrange))

    fig, ax = plt.subplots(figsize=(6, 6))
    plt.imshow(alldists_2d, extent=[-15,15,-15,15], origin='lower', cmap='viridis')
    plt.colorbar(label='Mean distance')
    plt.xlabel('x shift')
    plt.ylabel('y shift')
    plt.title('Mean nearest-neighbor distance for shifts')
    plt.show()


    # Apply translation to all cell centers
    X_final = X_new_rotated + best_shift
    print(best_shift)
    # Visualize the result
    fig, ax = plt.subplots(figsize=(6, 6))

    A2 = R.T @ R2.T    
    t3 = R2 @ np.array(t) - R2 @ np.array(center_of_rotation) + np.array(center_of_rotation) + np.array(best_shift[0])
    offset = -A2 @ t3


    img_transformed = affine_transform(
        img_gray,
        A2,
        offset=offset,
        order=1,               # bilinear interpolation
        mode="constant",       # zero padding
        cval=0.0
    )

    import matplotlib.patches as patches

    ax.imshow(img_transformed, cmap='gray')
    ax.scatter(
        old_cell_centers[:, 1], old_cell_centers[:, 0],
        color='blue', s=10, label='Old'
    )
    ax.scatter(
        X_final[:, 1], X_final[:, 0],
        color='red', s=10, label='New aligned'
    )
    # plot inliers with pink color
    ax.scatter(
        X_final[inliers, 1], X_final[inliers, 0],
        color='pink', s=20, label='Inliers'
    )

    from scipy.spatial import cKDTree
    import numpy as np

    # KD-tree for old cell centers
    tree_old = cKDTree(old_cell_centers)

    # Initialize masks
    not_has_1_neighbor = np.zeros(len(X_final), dtype=bool)
    has_1_neighbor = np.zeros(len(X_final), dtype=bool)

    # List to store old cell indices that are the single neighbor
    single_old_neighbors = []

    # Fill masks and record old indices
    for i in inliers:
        neighbors_idx = tree_old.query_ball_point(X_final[i], r=10)

        if len(neighbors_idx) == 1:
            has_1_neighbor[i] = True
            single_old_neighbors.append(neighbors_idx[0])  # record the old cell index
        else:
            not_has_1_neighbor[i] = True

    # Convert to NumPy array if needed
    single_old_neighbors = np.array(single_old_neighbors)

    #print("Indices of old cells that were single neighbors:", single_old_neighbors)


    ax.scatter(
        X_final[has_1_neighbor, 1], X_final[has_1_neighbor, 0],
        color='green', s=20, label='Only 1 Neighbor'
    )

    ax.scatter(
        old_cell_centers[single_old_neighbors, 1], old_cell_centers[single_old_neighbors, 0],
        color='orange', s=10, label='Old was 1 Neighbor'
    )
    
    # Draw red circles of radius 10 pixels around each inlier
    for y, x in X_final[not_has_1_neighbor]:  # note row/col order
        circle = patches.Circle((x, y), radius=10, edgecolor='red', facecolor='none', linewidth=1)
        ax.add_patch(circle)

    img2=ROIs.sum(0)
    img2_transformed = affine_transform(
        img2,
        A2,
        offset=offset,
        order=1,               # bilinear interpolation
        mode="constant",       # zero padding
        cval=0.0
    )
    #ax.imshow(img_transformed, cmap='gray', alpha=0.6)
    ax.imshow(img2_transformed, cmap='hot', alpha=0.4)

    OLD_ROIs = old_estimates['ROIs']

    img3=OLD_ROIs.sum(0)

    #ax.imshow(img_transformed, cmap='gray', alpha=0.6)
    ax.imshow(img3, cmap='viridis', alpha=0.2)

    ax.set_title('FINE TRANSLATION')
    ax.legend()
    plt.show()




Mean dist 14.206531565231716
Best shfit1: [-10  -2]
Mean dist 14.105084941385494
Best shfit1: [-10  -1]
Mean dist 14.117246444194532
Best shfit1: [-10   0]
Mean dist 13.934459304838601
Best shfit1: [-9 -3]
Mean dist 13.705227353427386
Best shfit1: [-9 -2]
Mean dist 13.602126336845881
Best shfit1: [-9 -1]
Mean dist 13.63081451975066
Best shfit1: [-9  0]
Mean dist 13.292556394634495
Best shfit1: [-8 -2]
Mean dist 13.178325954278577
Best shfit1: [-8 -1]
Mean dist 13.217734648620043
Best shfit1: [-8  0]
Mean dist 13.000862774381257
Best shfit1: [-7 -2]
Mean dist 12.905487718827963
Best shfit1: [-7 -1]
Mean dist 12.878768811275805
Best shfit1: [-6 -2]
Mean dist 12.782560448341142
Best shfit1: [-6 -1]
[[-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1]
 [-6 -1

In [ ]:

# # -----------------------------------------------------
# # ROI MATCHING + ID ASSIGNMENT (UNCHANGED LOGIC)
# # -----------------------------------------------------

#First>> Sort each cell into catergories based on number of neighbors in a 10 pixel radius

# KD-tree for old cell centers
tree_old = cKDTree(old_cell_centers)
tree_new = cKDTree(X_final)
ALL_OLD_ROIS = OLD_ROIs.sum(0)
OLD_ROIs_bool = np.array(OLD_ROIs, dtype=bool)  # (n_old, H, W)

# Initialize lists
has_0_neighbors = []
has_1_neighbor = []
has_2_or_more_neighbors = []

#inliers_set = set(inliers)
#outliers = [i for i in range(len(X_final)) if i not in inliers_set]

#Classify by neighbors
for idx in inliers:
    neighbors_idx = tree_old.query_ball_point(X_final[idx], r=10)

    if len(neighbors_idx) == 0:
        has_0_neighbors.append(idx)

    if len(neighbors_idx) == 1:
        has_1_neighbor.append(idx)

    if len(neighbors_idx) > 1:
        has_2_or_more_neighbors.append(idx)

CellIDs = np.zeros(len(ROIs)) #anything kept as zero will be assigned a new ID at the end

#Handle each list seperately
for idx in has_1_neighbor:
    new_neighbors_idx = tree_new.query_ball_point(X_final[idx], r=10)
    if len(new_neighbors_idx) == 0:
        neighbors_idx = tree_old.query_ball_point(X_final[idx], r=10)
        CellIDs[idx]= old_cell_IDs[neighbors_idx[0]]
    if len(new_neighbors_idx) > 0:
        neighbors_idx = tree_old.query_ball_point(X_final[idx], r=10)
        newROI = ROIs[idx]
        oldROI = OLD_ROIs[neighbors_idx[0]]
        intersection = np.logical_and(newROI, oldROI)
        union = np.logical_or(newROI, oldROI)
        overlap = np.sum(intersection) / np.sum(union)
        if overlap > 0.2:
            CellIDs[idx]= old_cell_IDs[neighbors_idx[0]]
        else:
            continue
    # could still have dups with two new cells assigned to same old cell

for idx in has_0_neighbors:
    newROI = ROIs[idx]
    intersection_area = np.sum(np.logical_and(newROI, ALL_OLD_ROIS))
    if intersection_area ==0:
        continue
    elif intersection_area >0:
        newROI = ROIs[idx].astype(bool)
        newROIarea = np.sum(newROI)
        intersection = OLD_ROIs_bool & newROI
        intersection_areas = intersection.sum(axis=(1, 2))
        overlaps = intersection_areas / newROIarea
        best_old_idx = np.argmax(overlaps)
        max_overlap = overlaps[best_old_idx]
        if max_overlap > 0.1: #if overlap is non-neglible
            if old_cell_IDs[best_old_idx] in CellIDs #test if matched already
                continue
            else: #test ROI overlap
                CellIDs[idx] = old_cell_IDs[best_old_idx] if max_overlap > 0.3 else 0
        else:
            continue


for idx in has_2_or_more_neighbors:
    neighbors_idx = tree_old.query_ball_point(X_final[idx], r=10)
    overlaps = []
    for old_idx in neighbors_idx:
        newROI = ROIs[idx]
        oldROI = OLD_ROIs[old_idx]
        intersection_area = np.sum(np.logical_and(newROI, oldROI))
        old_area = np.sum(oldROI)
        overlap = intersection_area/old_area
        overlaps.append(overlap)
    
    if max(overlaps) > 0.5:
        best_overlap_idx = np.argmax(overlaps)
        best_old_idx = neighbors_idx[best_overlap_idx] # corresponding index in old arrays
        CellIDs[idx] = old_cell_IDs[best_old_idx]
    else:
        continue
    

#go through CellIDs list and set new Cell IDs for any positions with zeros
largest_ID = max(old_cell_IDs.max(), 0)

for idx in range(len(ROIs)): #is this right??
    if CellIDs[idx] == 0:
        Cell_ID[idx] = largest_ID+1
        largest_ID = largest_ID+1
        print(f"New ROI {idx} is new, assigned ID {Cell_ID[idx]}")
    else:
        print(f"New ROI {idx} matches old ROI {CellIDs[idx]}")


In [ ]:


save_name = fname[:-4]+'volpy'
np.save(save_name, vpy.estimates)

cm.stop_server(dview=dview)
log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)

print("Saved VOLPY estimates to:", save_name + '.npy')

previous_folder_path = folder_path  # Update for next iteration


# print(vpy.estimates.keys())
# print(len(vpy.estimates['spikes']))
# #print(len(vpy.estimates['spikeTimes']))
# print(vpy.estimates['snr']) 

# #print length of each key's data:
# for key in vpy.estimates.keys():
#     print(f"{key}: {len(vpy.estimates[key])}")

# #print number of neurons with snr > 3
# snr_threshold = 3.0
# high_snr_neurons = np.sum(vpy.estimates['snr'] > snr_threshold)
# print(f"Number of neurons with SNR > {snr_threshold}: {high_snr_neurons}")



In [ ]:
# #load the saved VOLPY estimates for troubleshooting faster
# save_name = fname[:-4]+'volpy'
# estimates = np.load(save_name + '.npy', allow_pickle=True).item()
# vpy.estimates=estimates


In [65]:

##
vpynew = vpy.estimates
vpynew['spikes'] = np.array(vpynew['spikes'], dtype=object)

# try:


num_frames = np.max(vpynew['dFF'].shape)
dur = num_frames/640
vpynew['snr_over_3'] = []

vpynew['raster'] = np.zeros_like(vpynew['dFF'])
vpynew['firing_rate'] = np.zeros_like(vpynew['dFF'])
vpynew['unique_trace'] = []
vpynew['cell_idxs'] = []
# for i in range(vpynew['dFF'].shape[0]-1):
#     vpynew['raster'][i,vpynew['spikes'][i]] = 1
#     vpynew['firing_rate'][i] = savgol_filter(np.convolve(vpynew['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

#     if np.sqrt(np.var(vpynew['templates'][i], ddof=1))>0.5:
#         vpynew['cellID'].append(i)


In [ ]:

for i in range(vpynew['dFF'].shape[0]-1):
    vpynew['raster'][i, vpynew['spikes'][i]] = 1
    vpynew['firing_rate'][i] = savgol_filter(np.convolve(vpynew['raster'][i]*640,np.ones(32)/32,mode='same'),64,1)

for i in range(len(vpynew['Cell_IDs'])):
    vpynew['snr_over_3'].append(vpynew['snr'][i] > 3.0)

print("Number of neurons with SNR > 3:", np.sum(vpynew['snr_over_3']))

print(vpynew['Cell_IDs'])

Number of neurons with SNR > 3: 57


In [68]:

if np.sum(vpynew['snr_over_3']) > 0:
    to_remove = set()
    dFF = np.array(vpynew['dFF']).astype(float)
    R = np.corrcoef(dFF)
    idx0, idx1 = np.where(np.triu(R, 1) > 0.9)
    max_vals = np.max(dFF, axis=1)
    smaller = np.where(max_vals[idx0] < max_vals[idx1], idx0, idx1)
    to_remove.update(smaller.tolist())
    vpynew['unique_trace'] = [True if x not in to_remove else False for x in range(len(vpynew['Cell_IDs']))]

    # dFF = np.array(vpynew['dFF']).astype(float)
    # R = np.corrcoef(dFF)
    # r = np.array(np.where(np.triu(R,1)>0.7))
    # for i in range(0,r.shape[1]):
    #     if np.max(dFF[r[0][i]]) < np.max(dFF[r[1][i]]):
    #         r[1][i] = r[0][i]

    # vpynew['cellID'] = [x for x in vpynew['cellID'] if x not in r[1]]
print(vpynew['unique_trace'])
print("There are", np.sum(vpynew['unique_trace']), "unique traces after correlation filtering.")
print("And there were ", len(to_remove), "traces removed due to high correlation.")

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True]
There are 94 unique traces after correlation filtering.
And there were  2 traces removed due to high correlation.


In [69]:
vpynew['cell_idxs'] = []
for cell in range(len(vpynew['Cell_IDs'])):
    if vpynew['snr_over_3'][cell] and vpynew['unique_trace'][cell]:
        vpynew['cell_idxs'].append(cell)

print("Final number of cells after SNR and correlation filtering:", len(vpynew['cell_idxs']))
print(vpynew['cell_idxs'])
print(len(vpynew['cell_idxs']))

Final number of cells after SNR and correlation filtering: 56
[0, 1, 3, 4, 5, 6, 7, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 23, 24, 25, 27, 28, 30, 31, 32, 33, 34, 36, 37, 40, 41, 43, 46, 47, 48, 49, 53, 54, 56, 57, 58, 60, 63, 65, 66, 72, 73, 74, 75, 80, 83, 87, 91, 92, 95]
56


In [ ]:
#make figure

cells = np.array(vpynew['cell_idxs'])
time = np.arange(0,dur,1/640)

fig = plt.figure(figsize=(8.0, 11.0), facecolor='w',constrained_layout=True)
spec = fig.add_gridspec(ncols=3, nrows=5, width_ratios=[1,1,1], height_ratios=[2, 5,1,1,1])
ax1 = fig.add_subplot(spec[0, 0])
ax2 = fig.add_subplot(spec[0, 1])
ax_text = fig.add_subplot(spec[0, 2],facecolor='w')
ax3 = fig.add_subplot(spec[1, :],facecolor='w')
ax4 = fig.add_subplot(spec[4, :],facecolor='w')
ax5 = fig.add_subplot(spec[2, :],facecolor='w')
ax5r = ax5.twinx()
ax6 = fig.add_subplot(spec[3, :],facecolor='w')
#ax7 = fig.add_subplot(spec[4, :],facecolor='w')

ax1.imshow(img[:,:,1], cmap='gray')
ax2.imshow(img[:,:,2], cmap='gray')
ax1.set_title('Mean image',color='k',fontsize=14)
ax2.set_title('Corr image',color='k',fontsize=14)
ax1.set_axis_off()
ax2.set_axis_off()
ax_text.set_axis_off()

llim = 0
if len(cells)>0:
    pos_cells = []
    neg_cells = []
    b, a = butter(1, [1.5, 100], fs=640, btype='band')
    k = 1
    for i in range(0, len(cells)):
        if ''.join(vpynew['polarity'][cells[i]]) in 'negative':
            color = '#9AAB3A'
            mult = -1
            neg_cells.append(cells[i])
        else:
            color = '#54A0A8'
            mult = 1
            pos_cells.append(cells[i])
        y = np.array(lfilter(b,a,stats.zscore(np.array(vpynew['dFF'][cells[i]] * mult * 100,dtype=np.float32))) + ((k - 1) * 8)).reshape(1,num_frames)
        ax3.plot(llim+time,y[0,:],color, linewidth=0.3)
        ax3.plot(llim+time[vpynew['spikes'][cells[i]]],np.max(y)*np.ones(vpynew['spikes'][cells[i]].shape[0]),"|",color='firebrick',markersize=2)
        k = k + 1


    if len(pos_cells)>0:
        mean_fr_pos = np.mean(vpynew['firing_rate'][pos_cells,:], axis=0)
        sem_pos = stats.sem(np.array(vpynew['firing_rate'][pos_cells,:],dtype=np.float32), axis=0)
        ax5r.plot(llim+time, np.array(mean_fr_pos,dtype='float32').ravel(), label='Mean firing rate', color='#54A0A8',linewidth=0.3)
        ax5r.fill_between(llim+time, np.array(mean_fr_pos - sem_pos,dtype='float32').ravel(), np.array(mean_fr_pos + sem_pos,dtype='float32'), color='#54A0A8', alpha=0.3, label='SEM')
        ax5.set_ylabel('Firing rate (Hz)',color='#54A0A8',fontsize=12)
        ax5r.tick_params(axis ='y', labelcolor = '#54A0A8')
    if len(neg_cells)>0:
        mean_fr_neg = np.mean(vpynew['firing_rate'][neg_cells,:], axis=0)
        sem_neg = stats.sem(np.array(vpynew['firing_rate'][neg_cells,:],dtype=np.float32), axis=0)
        ax5.plot(llim+time, np.array(mean_fr_neg,dtype='float32').ravel(), label='Mean firing rate', color='#9AAB3A',linewidth=0.3)
        ax5.fill_between(llim+time, np.array(mean_fr_neg - sem_neg,dtype='float32').ravel(), np.array(mean_fr_neg + sem_neg,dtype='float32'), color='#9AAB3A', alpha=0.3, label='SEM')
        ax5.set_ylabel('Firing rate (Hz)',color='#9AAB3A',fontsize=12)
        ax5r.tick_params(axis ='y', labelcolor = '#9AAB3A')

wheel_mat = os.path.dirname(fname) + '\\Wheel.mat'
if os.path.exists(wheel_mat):
    wheel=mat73.loadmat(wheel_mat)
    if 'behavior' in wheel:
        ax4.plot(wheel['behavior'][:,0],wheel['behavior'][:,1],'r',linewidth=1.2)
        if wheel['behavior'].shape[1]>2:
            ax4.plot(wheel['behavior'][:,0],wheel['behavior'][:,2],'k',linewidth=1)
        ax4.set_ylabel('Behavior',color='k',fontsize=12)
        ax4.set_yticks([-1,0,1])
        ax4.set_ylim([-2,2])

    if wheel['data_time'].any():
        whl_time = np.arange(0,np.max(wheel['data_time']),1/640)
        wheel_interp = np.interp(whl_time, wheel['data_time'], wheel['data_pos'])
        speed = np.zeros_like(wheel_interp)
        for i in range(0,len(whl_time)-1):
            speed[i] = (wheel_interp[i+1]-wheel_interp[i])/(whl_time[i+1]-whl_time[i])

        #speed[speed>100] = 0
        #speed[speed<0] = 0
        speed = savgol_filter(speed,64,1)
        ax6.plot(whl_time,speed,'k',linewidth=1)
        ax6.set_ylabel('Speed (cm/s)',color='k',fontsize=12)
        #ax7.plot(whl_time,speed,'w',linewidth=1)
        #ax7.set_ylabel('Speed (cm/s)',color='k',fontsize=12)

    ax_text.text(0.5, 0.8, 'Mouse = ' + wheel['mouse'], color='k',fontsize=10, ha='center')
    ax_text.text(0.5, 0.4, 'Stimulus = ' + wheel['stimulus'], color='k',fontsize=10, ha='center')
    ax_text.text(0.5, 0.6, 'Date = ' + str(np.array(wheel['currentdate'],dtype='int32')), color='k',fontsize=10, ha='center')
    #ax_text.text(0.5, 0.2, 'File = ' + wheel['file'], color='w',fontsize=10, ha='center')
    if wheel['stimulus']=='Map' and 'rand_num' in wheel:
        ax_text.text(0.5, 0, 'Field = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
    elif wheel['stimulus']=='Tuning' and 'rand_num' in wheel:
        ax_text.text(0.5, 0, 'Orientation = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
    elif wheel['stimulus']=='Tuning' and 'rand_num' in wheel:
        ax_text.text(0.5, 0, 'Orientation = ' + " ".join(str(x) for x in wheel['rand_num'].astype(int)), color='k',fontsize=10, ha='center')
else:
    print("Wheel data does not exist")


for ax in [ax3,ax4,ax5,ax6]:
    ax.tick_params(color='black', labelcolor='black')
    ax.set_xlabel('Time (sec)',color='k',fontsize=12)
    ax.set_xlim([llim,llim+dur])
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
ax3.set_title('dFF',color='k',fontsize=14)
ax3.set_ylabel(r'$\Delta$F/F (%)',color='k',fontsize=12)


fig.savefig(fname[:-4] + '_volpy.pdf')
#plt.close('all')

print("Saved VOLPY figure to:", fname[:-4] + '_volpy.pdf')

print("Saving VOLPY data to MAT file...")
vpynew['ROIs'] = ROIs
#vpy['rect'] = r['rois']
vpynew['img'] = img
#del vpynew['rawROI']
#scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpynew': vpynew}, format='5', do_compression=True)





print("Converting data types for fast saving...")

# Keys identified from inspection output that need fixing
keys_to_convert_float = [
    't', 'ts', 't_rec', 't_sub', 'templates', 'snr', 
    'thresh', 'weights', 'locality', 'context_coord', 'F0', 'dFF', 
    'raster', 'firing_rate'
]

keys_to_convert_int = [
    'num_spikes'
]

# Process float conversions
for key in keys_to_convert_float:
    if key in vpynew and vpynew[key].dtype == object:
        try:
            # Attempt a direct conversion to float32 (fastest for scientific data)
            vpynew[key] = np.array(vpynew[key], dtype=np.float32)
            print(f"  Converted '{key}' to float32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to standard array dtype. Keeping as object array.")

# Process integer conversions
for key in keys_to_convert_int:
    if key in vpynew and vpynew[key].dtype == object:
        try:
            vpynew[key] = np.array(vpynew[key], dtype=np.int32)
            print(f"  Converted '{key}' to int32 array.")
        except ValueError:
            print(f"  Could not convert '{key}' to int32 array. Keeping as object array.")

# Handle variables that are inherently irregular lists that MUST be object arrays in Python, 
# but we ensure they are clean for saving.

# Handle 'mean_im', 'cell_n', 'polarity' (irregular shapes/strings)
for key in ['mean_im', 'cell_n', 'polarity']:
    if key in vpynew and vpynew[key].dtype == object:
        vpynew[key] = np.array(vpynew[key], dtype=object) # Ensure they are formally object arrays

# Handle spikes and low_spikes. The try/except handles the 'bool is not iterable' error.
if vpynew['spikes'].dtype == object:
    vpynew['spikes'] = np.array([list(x) for x in vpynew['spikes']], dtype=object)
    
if vpynew['low_spikes'].dtype == object:
    try:
        # This was causing the TypeError because it was actually a boolean array
        vpynew['low_spikes'] = np.array([list(x) for x in vpynew['low_spikes']], dtype=object)
    except TypeError:
        # If it's a bool array, just make sure it's saved as a clean boolean array
        vpynew['low_spikes'] = np.array(vpynew['low_spikes'], dtype=bool) 


print("Data type conversion complete.")

scipy.io.savemat(fname[:-4] + '_volpy.mat', {'vpynew': vpynew}, format='5', do_compression=True)
print("Saved VOLPY data to:", fname[:-4] + '_volpy.mat')


# vpynew.estimates['params'] = opts
# save_name = f'volpy_{os.path.split(fnames)[1][:-5]}_{threshold_method}'
# np.save(fnames[:-4] + '_volpy.npy', vpynew.estimates)

#del vpynew
# %% STOP CLUSTER and clean up log files

log_files = glob.glob('*_LOG_*')
for log_file in log_files:
    os.remove(log_file)
# except ValueError:
#     traceback.print_exc()
#     print("No volpy data was saved")

Saved VOLPY figure to: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1_volpy.pdf
Saving VOLPY data to MAT file...
Converting data types for fast saving...
  Converted 't' to float32 array.
  Converted 'ts' to float32 array.
  Converted 't_rec' to float32 array.
  Converted 't_sub' to float32 array.
  Converted 'templates' to float32 array.
  Converted 'snr' to float32 array.
  Converted 'thresh' to float32 array.
  Converted 'weights' to float32 array.
  Converted 'locality' to float32 array.
  Converted 'context_coord' to float32 array.
  Converted 'F0' to float32 array.
  Converted 'dFF' to float32 array.
  Converted 'raster' to float32 array.
  Converted 'firing_rate' to float32 array.
  Could not convert 'num_spikes' to int32 array. Keeping as object array.
Data type conversion complete.
Saved VOLPY data to: C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B\20250505\FOV1_T1\FOV1_T1_volpy.mat


In [ ]:
out_dir = os.path.dirname(fname)
out_path = os.path.join(out_dir, 'reconstructed_movie.mp4')

writer = imageio.get_writer(
    out_path,
    fps=40,
    codec='libx264',
    pixelformat='yuv420p'
)

for frame in mv_all:
    f = frame.astype(np.float32)
    f -= f.min()
    f /= f.max() + 1e-8
    f = (255 * f).astype(np.uint8)
    writer.append_data(f)

writer.close()

In [ ]:
import os
import imageio.v2 as imageio
import numpy as np

# --- parameters ---
orig_fps = 40          # original frame rate
seconds = 5
slowdown = 2.0         # 2x slower (increase for more slow-mo)

gif_fps = orig_fps / slowdown
n_frames = int(seconds * orig_fps)

# --- output path ---
out_dir = os.path.dirname(fname)
gif_path = os.path.join(out_dir, 'reconstructed_last5s_slow.gif')

# --- select last frames ---
frames = mv_all[-n_frames:]

with imageio.get_writer(
    gif_path,
    mode='I',
    fps=gif_fps,
    loop=0        # 0 = loop forever
) as writer:

    for frame in frames:
        f = frame.astype(np.float32)

        # crop to right third for visibility
        h, w = f.shape
        f = f[:, 2 * w // 3 : w]

        # normalize for visibility
        f -= f.min()
        f /= f.max() + 1e-8
        f = (255 * f).astype(np.uint8)

        writer.append_data(f)

print(f"Saved GIF to: {gif_path}")


In [ ]:
####SPIKINESS CODE

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# ===============================
# Parameters
# ===============================
TILE_SIZE = 4
H, W = 512, 512
FRAME_RATE = fr   # already defined
BANDPASS = BANDPASS       # (low, high), high ignored
DISPLAY_CLIP = 99         # percentile for visualization

# ===============================
# Robust spike metric
# ===============================
def spikiness_metric(trace, z_thresh=3.0):
    """
    Estimate amount of outlying activity in a trace.
    Uses robust z-score (MAD-based) and measures tail mass.
    """
    trace = trace - np.median(trace)
    mad = np.median(np.abs(trace)) + 1e-9
    z = trace / mad

    # Fraction of samples beyond threshold
    spike_fraction = np.mean(np.abs(z) > z_thresh)

    # Mean excess magnitude beyond threshold
    spike_energy = np.mean(np.abs(z[np.abs(z) > z_thresh])) if np.any(np.abs(z) > z_thresh) else 0.0

    return spike_fraction + 0.1 * spike_energy


# ===============================
# Output map (tile-resolution)
# ===============================
n_tiles_y = H // TILE_SIZE
n_tiles_x = W // TILE_SIZE

tile_spike_map = np.zeros((n_tiles_y, n_tiles_x), dtype=np.float32)

# ===============================
# Main loop
# ===============================
with tqdm(total=n_tiles_y * n_tiles_x, desc="Analyzing tiles") as pbar:
    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):

            y0 = ty * TILE_SIZE
            y1 = y0 + TILE_SIZE
            x0 = tx * TILE_SIZE
            x1 = x0 + TILE_SIZE

            # Extract tile: shape (T, 16, 16)
            tile = video[:, y0:y1, x0:x1]

            # Sum all pixels
            tile_trace = tile.reshape(T, -1).sum(axis=1)

            # High-pass filter (bandpass-compatible)
            tile_trace_filt = bandpass_filter(
                tile_trace, FRAME_RATE, *BANDPASS
            )

            # Compute spikiness
            tile_spike_map[ty, tx] = spikiness_metric(tile_trace_filt)

            pbar.update(1)

# ===============================
# Expand tile map back to image resolution
# ===============================
spike_image = np.repeat(
    np.repeat(tile_spike_map, TILE_SIZE, axis=0),
    TILE_SIZE, axis=1
)

# # ===============================
# # Visualization
# # ===============================
# vmax = np.percentile(spike_image, DISPLAY_CLIP)

# plt.figure(figsize=(6, 6))
# plt.imshow(spike_image, cmap="hot", vmin=0, vmax=vmax)
# plt.title("Grid-based spike activity ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
# plt.colorbar(label="Spikiness score")
# plt.axis("off")
# plt.tight_layout()
# plt.show()

# ===============================
# Visualization (normalized)
# ===============================

# Normalize spike image to [0, 1]
spike_norm = spike_image.astype(np.float32)
spike_norm -= spike_norm.min()
spike_norm /= (spike_norm.max() + 1e-9)  # avoid division by zero

# Optionally clip at DISPLAY_CLIP percentile for contrast
vmax = np.percentile(spike_norm, DISPLAY_CLIP / 100 * 1.0)

plt.figure(figsize=(6, 6))
plt.imshow(spike_norm, cmap="hot", vmin=0, vmax=vmax)
plt.title(f"Grid-based spike activity ({TILE_SIZE}×{TILE_SIZE} tiles)")
plt.colorbar(label="Spikiness score")
plt.axis("off")
plt.tight_layout()
plt.show()

# ===============================
# Visualization
# ===============================
vmax = np.percentile(spike_image, DISPLAY_CLIP)

plt.figure(figsize=(6, 6))
plt.imshow(spike_image, cmap="hot", vmin=0, vmax=vmax)
plt.title("Grid-based spike activity ("+str(TILE_SIZE)+"×"+str(TILE_SIZE)+" tiles)")
plt.colorbar(label="Spikiness score")
plt.axis("off")
plt.tight_layout()
plt.show()